# Cropchain Phase 2 — Scrum Planning
## Experiment 2: Sprint Execution — Development Tasks, QA Coverage, and UI/UX Design

**Project:** Cropchain: Intelligent Farm-to-Table Supply Chain Management System  
**Methodology:** Phase 2 — Agile Scrum  
**Author:** *[Replace with your full name]*  
**Course:** CS587 — Software Project Management  

---

## 📋 Professional Cleanup Requirements — Read This First

Before submitting this notebook, you are required to:

1. **Remove all instruction comments** from code cells — any line starting with `# TODO` or `# INSTRUCTIONS` must be deleted. The final notebook must read as a clean, professional deliverable with no scaffolding visible.
2. **Fill in your full name** in the Author field above.
3. **Write all analysis sections in your own words.** Every markdown section marked with *[your analysis here]* must be completed — no placeholder text in the final submission.
4. **Run all cells top to bottom** before submitting so every code cell has real output visible below it.
5. **Delete this entire `📋 Professional Cleanup Requirements` section** from your final submitted version — it is a working instruction, not part of the deliverable.
6. **Read your agent outputs critically.** After running the agents, evaluate whether their responses are accurate, coherent, and specific to Cropchain. Your Section 9 analysis must reflect this genuine review.
7. **Ensure consistent structure throughout:** each section must have a markdown explanation before its code cell and a written interpretation or analysis afterward where indicated.

---

## Experiment Overview

This notebook is your individual contribution to Phase 2 of the Cropchain final project. It picks up directly from the Team Lead's Experiment 1, which produced the Product Backlog and Sprint 1 Plan using the Scrum framework.

Your responsibility in this experiment is to simulate the **Sprint execution planning** stage of Scrum. Three specialized agents are activated here, each taking on a role from the Scrum development team:

- **Developer Agent** — translates Sprint 1 user stories into concrete technical tasks with role assignments and hour-level effort estimates
- **QA Engineer Agent** — produces test cases mapped to both user story IDs and Phase 1 requirement IDs, covering functional and non-functional testing
- **UI/UX Designer Agent** — contributes interface design details for all user-facing Sprint 1 stories

The Scrum Master agent coordinates all three, consistent with the role defined in Experiment 1.

### Why This Experiment Matters for the Project

The outputs you generate here are loaded directly into Teammate 2's Experiment 3. They will be used to build the DevOps pipeline plan, Sprint Review simulation, documentation plan, and the final Scrum vs Waterfall comparative analysis. **If your output files are missing or incomplete, Teammate 2's notebook will fail at startup.**

### Prerequisites — Confirm Before Running

The Team Lead must have already run Experiment 1. Verify that these two files exist before proceeding:
- `src/outputs/phase2/experiment_1/03_product_backlog.md`
- `src/outputs/phase2/experiment_1/04_sprint1_plan.md`

### Git Instructions
```bash
# Switch to the Phase 2 working branch and pull the latest
git checkout phase2
git pull origin phase2

# After finishing and cleaning up your notebook:
git add 05_Cropchain_Phase2_Experiment2_Dev_and_QA.ipynb
git commit -m "Phase 2 Exp 2: [Your Name] - Developer tasks, QA coverage, UI/UX design"
git push origin phase2
```

## Section 1: Environment Setup and Artifact Loading

The environment configuration is identical to Phase 1 and the Team Lead's Experiment 1 notebook. The same model (`gpt-4o-mini`), the same `TEAM_LLM_CONFIG`, and the same `.env` file are used across all team notebooks to ensure consistency and reproducibility.

This section also loads the two upstream artifacts produced by the Team Lead. These files are injected as context into every agent prompt in this notebook so that the Developer, QA, and Designer agents all work from the actual Sprint 1 scope — not from a generic or invented starting point.

If either file is missing, the cell will raise a `FileNotFoundError` with a message telling you what to do.

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
import autogen

CURRENT_DIR = Path.cwd()
BASE_DIR = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR

load_dotenv(BASE_DIR / ".env", override=True)
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY is missing. Add it to your .env file.")

MODEL_NAME = "gpt-4o-mini"
TEAM_LLM_CONFIG = {
    "config_list": [{"model": MODEL_NAME, "api_key": OPENAI_API_KEY}],
    "temperature": 0.2,
    "timeout": 120,
}

PROJECT_NAME = "Cropchain: Intelligent Farm-to-Table Supply Chain Management System"

# Directories
PHASE2_EXP1_DIR  = BASE_DIR / "src" / "outputs" / "phase2" / "experiment_1"
OUTPUT_DIR       = BASE_DIR / "src" / "outputs" / "phase2" / "experiment_2"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Load upstream artifacts from Team Lead's Experiment 1
BACKLOG_FILE      = PHASE2_EXP1_DIR / "03_product_backlog.md"
SPRINT1_PLAN_FILE = PHASE2_EXP1_DIR / "04_sprint1_plan.md"

for f in [BACKLOG_FILE, SPRINT1_PLAN_FILE]:
    if not f.exists():
        raise FileNotFoundError(
            f"Required Team Lead file missing: {f}\n"
            "Ask the Team Lead to run Experiment 1 and push to phase2 branch first."
        )

product_backlog_text = BACKLOG_FILE.read_text(encoding="utf-8")
sprint1_plan_text    = SPRINT1_PLAN_FILE.read_text(encoding="utf-8")

print("Environment loaded successfully.")
print("Model:", MODEL_NAME)
print("Output directory:", OUTPUT_DIR)
print("Product Backlog loaded:", bool(product_backlog_text.strip()))
print("Sprint 1 Plan loaded:",   bool(sprint1_plan_text.strip()))

## Section 2: Agent Definitions

Four agents are used in this experiment. The **Scrum Master** coordinates and initiates all conversations. The three **specialist agents** — Developer, QA Engineer, and UI/UX Designer — each respond with structured, Cropchain-specific outputs.

The Scrum Master's system message is provided below as a working reference example. Study it carefully — pay attention to how it defines the role, lists responsibilities, and ends with a formatting instruction. Your three agent system messages should follow the same level of detail and specificity.

**Your task:** Write the `system_message` for the Developer Agent, QA Engineer Agent, and Designer Agent. Each agent must know:
- What their specific Scrum role is on the Cropchain team
- Exactly what outputs they are expected to produce (numbered, specific)
- Any productivity assumptions that apply to them
- That their output must be Cropchain-specific — not generic software planning
- The shared technology stack: React.js (frontend), Node.js + Express (backend), PostgreSQL (database)

> **Reference:** Scrum team roles — https://www.atlassian.com/agile/scrum/roles

In [ ]:
# Scrum Master — provided as a complete reference. Do not modify.
scrum_master_agent = autogen.ConversableAgent(
    name="Scrum_Master",
    system_message="""
You are the Scrum Master for the Cropchain Scrum project.
In this experiment you coordinate the development team.
Route tasks to the correct team members and ensure outputs stay consistent
with the Sprint 1 Plan and Product Backlog.
Keep outputs structured and presentation-ready.
""".strip(),
    llm_config=TEAM_LLM_CONFIG,
    code_execution_config=False,
    human_input_mode="NEVER",
)

# TODO: Write the Developer Agent system message.
# This agent is a Full-Stack Developer on the Cropchain Scrum team.
# It must be instructed to:
#   - Break each Sprint 1 user story into 3-5 concrete technical tasks
#   - Assign each task a role: Backend / Frontend / Database / AI Service / DevOps
#   - Estimate task effort in hours (assume 6 productive hours per developer per day)
#   - Identify technical dependencies between tasks
#   - Flag tasks that address non-functional requirements: performance (<2s), scalability (10k users), uptime (99.9%), GDPR
#   - Include review and rework tasks per story
#   - Use the PostgreSQL + React.js + Node.js stack
developer_agent = autogen.ConversableAgent(
    name="Developer_Agent",
    system_message="""
# TODO: Replace this with your Developer Agent system message.
""".strip(),
    llm_config=TEAM_LLM_CONFIG,
    code_execution_config=False,
    human_input_mode="NEVER",
)

# TODO: Write the QA Engineer Agent system message.
# This agent is the QA / Test Engineer on the Cropchain Scrum team.
# It must be instructed to:
#   - Create 2-3 test cases per Sprint 1 user story
#   - Map each test case to its story ID (US-XXX) and requirement ID (REQ-XXX)
#   - Define test type: Unit / Integration / System / Acceptance
#   - Write test steps and expected results
#   - Cover Cropchain-specific features: yield prediction, fair pricing, alerts, traceability
#   - Confirm each story's acceptance criteria are testable
#   - Provide a total test case count and effort estimate (productivity: 2 test cases per day)
qa_engineer_agent = autogen.ConversableAgent(
    name="QA_Engineer",
    system_message="""
# TODO: Replace this with your QA Engineer Agent system message.
""".strip(),
    llm_config=TEAM_LLM_CONFIG,
    code_execution_config=False,
    human_input_mode="NEVER",
)

# TODO: Write the UI/UX Designer Agent system message.
# This agent is the UI/UX Designer on the Cropchain Scrum team.
# It must be instructed to:
#   - Describe the wireframe layout for each UI-relevant Sprint 1 story (plain text, no image)
#   - Note accessibility considerations per story
#   - Identify UX pain points the design addresses for Cropchain users
#   - Estimate design effort in hours (assume 4 productive design hours per day)
#   - Confirm mobile responsiveness is addressed for every UI story (REQ-013)
#   - Skip backend-only stories that have no user interface
designer_agent = autogen.ConversableAgent(
    name="Designer_Agent",
    system_message="""
# TODO: Replace this with your UI/UX Designer Agent system message.
""".strip(),
    llm_config=TEAM_LLM_CONFIG,
    code_execution_config=False,
    human_input_mode="NEVER",
)

print("All Experiment 2 agents created successfully.")

## Section 3: Prompt Design

Each prompt is sent by the Scrum Master to one specialist agent. All three prompts inject `product_backlog_text` and `sprint1_plan_text` as context so the agents work from the actual Cropchain Sprint 1 scope rather than generating generic content.

**Your task:** Write all three prompts below as f-strings. A well-constructed prompt for this project:
- Opens by establishing what has already been completed (Sprint 1 planning is done)
- Includes labeled context sections using the upstream variables already loaded
- Gives clear, numbered output requirements specific to the agent's role
- References Cropchain domain specifics (actors, features, non-functional requirements)
- Ends with a formatting instruction: structured and presentation-ready

Study the prompts in the Team Lead's Experiment 1 notebook and the Phase 1 notebooks as reference before writing yours.

In [ ]:
# TODO: Write the prompt from the Scrum Master to the Developer Agent.
# Requirements for this prompt:
#   - Open by stating Sprint 1 planning for Cropchain is complete
#   - Include product_backlog_text and sprint1_plan_text as labeled context sections
#   - Ask for a technical task breakdown per Sprint 1 story
#   - Specify: 3-5 tasks per story, role assignment, hour estimates, dependencies
#   - Mention the tech stack: React.js, Node.js + Express, PostgreSQL
#   - Request non-functional requirement notes and review/rework tasks
#   - End with: "Make the output structured and presentation-ready."
sm_to_developer_prompt = f"""
# TODO: Write your Developer prompt here as an f-string.
# Use {{product_backlog_text}} and {{sprint1_plan_text}} to inject context.
""".strip()

# TODO: Write the prompt from the Scrum Master to the QA Engineer.
# Requirements for this prompt:
#   - Open by stating Sprint 1 planning for Cropchain is complete
#   - Include product_backlog_text and sprint1_plan_text as labeled context sections
#   - Ask for 2-3 test cases per Sprint 1 story, mapped to story IDs and requirement IDs
#   - Require coverage of: yield prediction, fair pricing, alerts, traceability, shipment visibility
#   - Request total test case count and effort estimate (2 test cases per day)
#   - End with: "Make the output structured and presentation-ready."
sm_to_qa_prompt = f"""
# TODO: Write your QA prompt here as an f-string.
# Use {{product_backlog_text}} and {{sprint1_plan_text}} to inject context.
""".strip()

# TODO: Write the prompt from the Scrum Master to the Designer Agent.
# Requirements for this prompt:
#   - Open by stating Sprint 1 planning for Cropchain is complete
#   - Include product_backlog_text and sprint1_plan_text as labeled context sections
#   - Ask for wireframe descriptions for all UI-relevant Sprint 1 stories
#   - Request: accessibility notes, UX pain points, effort estimates per story
#   - Instruct the agent to skip backend-only stories
#   - Require mobile responsiveness confirmation per story (REQ-013)
#   - End with: "Make the output structured and presentation-ready."
sm_to_designer_prompt = f"""
# TODO: Write your Designer prompt here as an f-string.
# Use {{product_backlog_text}} and {{sprint1_plan_text}} to inject context.
""".strip()

print("Prompts prepared.")
print("Developer prompt preview (first 300 chars):")
print(sm_to_developer_prompt[:300] + "\n...")

## Section 4: Output Persistence Helpers

These helper functions are shared across all Phase 2 notebooks. They save agent outputs as markdown files and raw chat histories to the experiment output directory. The three markdown files produced here are the artifacts that Teammate 2 will load in Experiment 3.

In [ ]:
def save_text(filename: str, text: str):
    """Save a string to a file in the experiment output directory."""
    file_path = OUTPUT_DIR / filename
    file_path.write_text(text, encoding="utf-8")
    print(f"Saved: {file_path}")

def save_chat_history(filename: str, chat_result):
    """Save the full AutoGen chat history to a readable text file."""
    file_path = OUTPUT_DIR / filename
    if hasattr(chat_result, "chat_history"):
        lines = []
        for item in chat_result.chat_history:
            role = item.get("name", item.get("role", "Unknown"))
            content = item.get("content", "")
            lines.append(f"{role}:\n{content}\n")
        file_path.write_text(("\n" + ("-" * 80) + "\n").join(lines), encoding="utf-8")
        print(f"Saved chat history: {file_path}")
    else:
        print("No chat_history found on chat result.")

def extract_last_message(chat_result) -> str:
    """Extract the final agent response from a completed chat result."""
    if hasattr(chat_result, "chat_history") and chat_result.chat_history:
        return chat_result.chat_history[-1].get("content", "")
    return ""

print("Output helpers defined.")

## Section 5: Run Experiment 2

This section executes the three-step agent workflow sequentially:

1. **Scrum Master → Developer Agent** — technical task breakdown for all Sprint 1 stories
2. **Scrum Master → QA Engineer** — test cases mapped to story and requirement IDs
3. **Scrum Master → Designer Agent** — UI/UX design notes for interface-facing stories

Each conversation uses `max_turns=1` — one prompt, one structured response. This is the same pattern used throughout the project.

**Your task:** Complete the three agent calls inside `run_phase2_experiment_2()`. Use `initiate_chat()` with the correct `recipient`, `message`, and `max_turns` arguments. Look at the Team Lead's Experiment 1 notebook for the exact call pattern.

In [ ]:
def run_phase2_experiment_2():

    print("=" * 80)
    print("STEP 1: Scrum Master → Developer Agent")
    print("=" * 80)
    # TODO: Replace None with your initiate_chat call.
    # Use: scrum_master_agent.initiate_chat(recipient=..., message=..., max_turns=1)
    sm_to_dev_result = None

    print("\n" + "=" * 80)
    print("STEP 2: Scrum Master → QA Engineer")
    print("=" * 80)
    # TODO: Replace None with your initiate_chat call.
    # Use: scrum_master_agent.initiate_chat(recipient=..., message=..., max_turns=1)
    sm_to_qa_result = None

    print("\n" + "=" * 80)
    print("STEP 3: Scrum Master → Designer Agent")
    print("=" * 80)
    # TODO: Replace None with your initiate_chat call.
    # Use: scrum_master_agent.initiate_chat(recipient=..., message=..., max_turns=1)
    sm_to_designer_result = None

    return sm_to_dev_result, sm_to_qa_result, sm_to_designer_result

sm_to_dev_result, sm_to_qa_result, sm_to_designer_result = run_phase2_experiment_2()

## Section 6: Save Experiment 2 Artifacts

The outputs from each agent are saved as markdown files to `src/outputs/phase2/experiment_2/`. These three files are required inputs for Teammate 2's Experiment 3 — without them, Experiment 3 will fail to load at startup. Once this section runs without errors, notify Teammate 2 that their inputs are ready.

In [ ]:
save_chat_history("01_sm_to_developer_chat.txt", sm_to_dev_result)
save_chat_history("02_sm_to_qa_chat.txt",        sm_to_qa_result)
save_chat_history("03_sm_to_designer_chat.txt",   sm_to_designer_result)

dev_output      = extract_last_message(sm_to_dev_result)
qa_output       = extract_last_message(sm_to_qa_result)
designer_output = extract_last_message(sm_to_designer_result)

if dev_output:      save_text("04_developer_tasks.md",  dev_output)
if qa_output:       save_text("05_qa_test_cases.md",     qa_output)
if designer_output: save_text("06_design_notes.md",      designer_output)

print("\nPhase 2 Experiment 2 artifacts saved.")

## Section 7: Artifact Validation

This cell confirms that all three output files were successfully written. All three must show `FOUND` before you push to the branch or notify Teammate 2.

In [ ]:
required_files = [
    OUTPUT_DIR / "04_developer_tasks.md",
    OUTPUT_DIR / "05_qa_test_cases.md",
    OUTPUT_DIR / "06_design_notes.md",
]

all_found = True
for f in required_files:
    status = "FOUND" if f.exists() else "MISSING"
    print(f"{f.name}: {status}")
    if status == "MISSING":
        all_found = False

if all_found:
    print("\nAll artifacts validated. Notify Teammate 2 that Experiment 3 inputs are ready.")
else:
    print("\nWARNING: Fix missing files before notifying Teammate 2.")

## Section 8: Display Generated Outputs

This section renders all three generated markdown files inline for review. Read through each output carefully before completing your analysis in Section 9. Your evaluation must reflect what the agents actually produced — not what you expected them to produce.

In [ ]:
from IPython.display import Markdown, display

for filename, label in [
    ("04_developer_tasks.md",  "DEVELOPER TECHNICAL TASK BREAKDOWN"),
    ("05_qa_test_cases.md",    "QA TEST CASES"),
    ("06_design_notes.md",     "UI/UX DESIGN NOTES"),
]:
    path = OUTPUT_DIR / filename
    if path.exists():
        print("=" * 60)
        print(label)
        print("=" * 60)
        display(Markdown(path.read_text(encoding="utf-8")))

## Section 9: Experiment Analysis and Evaluation

**This entire section must be written by you in your own words after running the experiment. Delete all placeholder text before submitting.**

---

### 9.1 What This Experiment Produced

*[Write 3-5 sentences summarizing what the three agents generated. Be specific — mention actual story IDs, task roles, or test case types that appeared in your output. Do not write a generic description.]*

---

### 9.2 Developer Agent — Accuracy Assessment

*[Evaluate the Developer Agent's technical task breakdown. Did the tasks align with the actual Sprint 1 stories? Were hour estimates reasonable for a 6-hour productive day? Were the right roles (Backend, Frontend, Database, AI Service) assigned? Did the agent correctly reference the PostgreSQL and React.js stack from Phase 1? Point to specific examples from the output.]*

---

### 9.3 QA Engineer — Coverage Assessment

*[Evaluate the QA Engineer's output. Are all Sprint 1 stories covered? Do the test cases correctly map to both story IDs (US-XXX) and requirement IDs (REQ-XXX)? Are Cropchain-specific features — yield prediction, fair pricing, traceability, alerts — tested? Was the effort estimate (2 test cases/day) applied correctly? What is missing or incomplete?]*

---

### 9.4 UI/UX Designer — Design Quality Assessment

*[Evaluate the Designer Agent's output. Are the wireframe descriptions meaningful and specific to Cropchain's users (Farmer, Buyer, Admin)? Are accessibility considerations genuine, not generic? Is mobile responsiveness addressed (REQ-013)? Which stories were skipped as backend-only, and was that correct?]*

---

### 9.5 Coherence with Phase 1 and Experiment 1

*[Discuss whether this experiment's outputs are consistent with the broader project. Do the developer tasks use the same technology stack defined in Phase 1 Experiment 2 (PostgreSQL, React.js, Node.js)? Do the QA test cases reference the same requirement IDs (REQ-001 to REQ-017) established in Phase 1 Experiment 1? Does the Sprint 1 scope in your outputs match what the Team Lead's Scrum Master committed to?]*

---

### 9.6 Limitations

*[Acknowledge at least 3 real limitations of this experiment. Consider: are hour estimates verified or assumed? Can plain-text wireframe descriptions replace actual design mockups? What would a real sprint execution reveal that this simulation cannot? Are the test cases executable as written, or do they require more detail?]*

---

### 9.7 Conclusion

*[Write 3-4 sentences concluding this experiment. State what was achieved, why it matters for the Cropchain Phase 2 project, and what Teammate 2 can now do with the outputs you produced.]*